# Qwen3-4B hybrid checkpoint — thinking-mode manual pilot
Use a fresh T4 runtime and the new hybrid package. This replaces Thinking-2507; it does not migrate the platform or disable thinking.
Follow docs/QWEN3_HYBRID_MANUAL_RUN.md. One neutral GPU smoke precedes the clean autonomous task; do not repeat the old diagnostic campaign.
The checkpoint’s native template leaves the opening thinking marker for the model to generate. No opening/closing marker is fabricated. Constrained JSON follows natural closure, as a recorded adaptation.
P0/R1 are engineering positions, not the original report-content P1. All previous runs remain separate.


In [ ]:
from google.colab import files
from pathlib import Path
import hashlib, json, sys, subprocess, zipfile
uploaded = files.upload()  # work/qwen3_hybrid_manual_20261101.zip
assert len(uploaded) == 1
name, archive_bytes = next(iter(uploaded.items()))
assert name.endswith('.zip')
print('Archive SHA256:', hashlib.sha256(archive_bytes).hexdigest())
root_parent = Path('/content/qwen3_hybrid_capstone')
assert not root_parent.exists(), 'Use a fresh runtime; do not overlay an existing package'
with zipfile.ZipFile(name) as z:
    for entry in z.infolist():
        path = Path(entry.filename)
        assert not path.is_absolute() and '..' not in path.parts
    z.extractall(root_parent)
ROOT = root_parent / 'ai-safety-capstone'
manifest = json.loads((ROOT/'PACKAGE_MANIFEST.json').read_text())
for relative, expected in manifest['files_sha256'].items():
    assert hashlib.sha256((ROOT/relative).read_bytes()).hexdigest() == expected, relative
print('Verified package files:', len(manifest['files_sha256']))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT/'requirements-thinking.txt')], check=True)
sys.path.insert(0, str(ROOT/'src'))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from datetime import datetime, timezone
import torch
assert torch.cuda.is_available(), 'Select GPU runtime first'
assert 'T4' in torch.cuda.get_device_name(0), 'This condition is prepared for T4; record/review a different GPU separately'
save_dir = Path('/content/drive/MyDrive/ai-safety-capstone') / ('qwen3_hybrid_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
save_dir.mkdir(parents=True, exist_ok=False)
(save_dir/'PACKAGE_MANIFEST.json').write_text(json.dumps(manifest, indent=2))
(save_dir/'archive_sha256.txt').write_text(hashlib.sha256(archive_bytes).hexdigest())
from qwen_adapter.settings import validate
config = validate(json.loads((ROOT/'configs/precommit_hook_qwen3_hybrid_clean.json').read_text()))
from qwen_adapter.thinking_model import ThinkingQwen
engine = ThinkingQwen(config['agent']['model'], config['agent']['qwen_options']['generation'])
assert engine.metadata['resolved_revision'] == config['agent']['qwen_options']['generation']['revision']
assert engine.model_id == 'Qwen/Qwen3-4B'
assert engine.metadata['native_thinking_flag'] is True
assert engine.metadata['template_sha256'] == 'a55ee1b1660128b7098723e0abcd92caa0788061051c62d51cbe87d9cf1974d8'
(save_dir/'model_metadata.json').write_text(json.dumps(engine.metadata, indent=2))
(save_dir/'config.json').write_text(json.dumps(config, indent=2))
print(json.dumps(engine.metadata, indent=2))
print('Evidence:', save_dir)


## GPU-only smoke: one generation, no shell execution
This validates reasoning/action generation and online-versus-replay vectors at short input length. It does not establish that a long context fits. Save failure evidence and stop if it fails; do not press repeatedly to obtain a pass.


In [ ]:
from qwen_adapter.manual_exchange import generate_bundle
from qwen_adapter.protocol import parse_generation
from qwen_adapter.spool import digest
import uuid

def evaluate_and_save(request):
    cid = request['call_id']
    bundle = generate_bundle(request, engine, Path('/content/qwen3_hybrid_records'), retained_dir=save_dir/'records')
    path = Path('/content') / (cid + '.response_bundle.json')
    serialized = json.dumps(bundle, ensure_ascii=False, indent=2)
    path.write_text(serialized)
    (save_dir/path.name).write_text(serialized)
    print('Call:', cid)
    print('Error:', bundle['response'].get('error'))
    print('Output:', bundle['response'].get('text'))
    raw = bundle.get('generation') or {}
    print('Tokens:', len(raw.get('input_token_ids', [])), len(raw.get('generated_token_ids', [])))
    print('Seconds:', raw.get('elapsed_seconds'), 'peak allocated bytes:', raw.get('peak_allocated_bytes'))
    files.download(str(path))  # download failures too
    if bundle['response'].get('error'):
        raise RuntimeError('STOP: preserve this bundle and diagnose; do not retry')
    parsed = parse_generation(raw, engine.settings)
    for position, capture in raw['activations'].items():
        print(position, 'replay:', capture['replay_passed'], 'dimensions:', {k:len(v) for k,v in capture['vectors'].items()})
    return bundle, parsed

if 'smoke_request' not in globals():
    smoke_request = {'call_id':uuid.uuid4().hex, 'model':engine.model_id, 'settings':engine.settings,
        'messages':[{'role':'system','content':'You are testing a reasoning interface. Do not execute shell commands.'},
                    {'role':'user','content':'What is 7 + 8?'}],
        'tools':[{'type':'function','function':{'name':'execute_command','description':'Execute a shell command on the Mac.',
            'parameters':{'type':'object','properties':{'command':{'type':'string'}},'required':['command']}}}]}
    smoke_request['request_hash'] = digest(smoke_request)
smoke_bundle, smoke_parsed = evaluate_and_save(smoke_request)
assert smoke_parsed['kind'] == 'report', 'STOP: smoke must finish without tools'
print('Short-input GPU pipeline passed. Review the report, then follow the separate clean-task launch instructions; do not repeat the old integration suite.')


## Repeat for each exported Mac request
Upload exactly one request. Check its call ID against Terminal B. Do not upload an earlier request from Downloads. Importing the downloaded bundle on your Mac releases the waiting controller and may execute the chosen command. Keep this notebook free of manual coaching between turns.


In [ ]:
uploaded_request = files.upload()
assert len(uploaded_request) == 1
request_name, request_bytes = next(iter(uploaded_request.items()))
request = json.loads(request_bytes)
bundle, parsed = evaluate_and_save(request)
print('Import this call bundle on Mac:', request['call_id'])


Stop on any generation, protocol, replay, memory or task failure. Preserve bundles and Drive records; no automatic retry, cap escalation or further prompt variants.
If the clean task fails, review its failure mechanism before deciding on a stronger model and larger GPU. Do not provision paid infrastructure or pool model conditions automatically.
